## 装饰器
一句话描述： 在不修改原函数代码的情况下，为函数增加额外的行为。

“@” 只是一个语法糖

### 先用代码理解装饰器
实际上装饰器就是一个闭包函数。

In [5]:


# 定义一个装饰器
def add_logger(fun):
  ''' 这个装饰器作用： 给普通函数执行前后添加上打印日志。'''
  def inner():
    print('函数执行前')
    fun()
    print('函数执行后')

  return inner


# 一个普通函数
def hello():
  print('hello python')


# 给普通函数加上装饰器
hello = add_logger(hello)

# 调用时，就会调用被装饰器加强后的函数
hello()


函数执行前
hello python
函数执行后


### 再尝试 @ 语法糖

可以看到，下面的代码和上面的输出是一致的。

```python
@add_logger
def hello_two():
  print('hello xxx')
```

所以，上面代码在背后做了两件事情：
1. 调用 add_logger 函数，并把 hello_two 函数作为参数传递进去
2. 把 add_logger 函数的返回值（内部函数），赋值给 hello_two 变量。



In [4]:
@add_logger
def hello_two():
  print('hello python twice!')


hello_two()

函数执行前
hello python twice!
函数执行后


## 第一次优化
支持参数调用

In [24]:

def add_logger(fn):
  def inner(msg: str):
    print('before log')
    fn(msg)
    print('after log')
  return inner


@add_logger
def hello(msg: str):
  print(f'hello {msg}')

hello('ellis')




before log
hello ellis
after log


### 学习 *args 和 **kwargs


In [ ]:
def test_two_stars(*args, **kwargs):
    """
    测试一下这个 *args / **kwargs 语法
    - *args: 解构赋值， 类似 js 中的 ...rest ，支持任意数量的位置参数
    - **kwargs: 接受任意数量的关键字参数。
    """
    print(f"args={args}, type={type(args)}")
    print(f"kwargs={kwargs}")
    for item in args:
        print(f"{item}")


test_two_stars(
    {"name": "ellis", "age": 18}, "男", study_hard=True, job="LLM Agent Engineer"
)

## 再试试把 *args 和 **kwargs 放进去

In [33]:


import time


def add_time_counter(fn):
  '''记录函数调用耗时情况'''
  def inner(*args, **kwargs):
    try:
      start = time.perf_counter()
      fn(*args, **kwargs)
      use_time = time.perf_counter() - start
      print(f'执行函数耗时：{use_time:.2f}s')
    except Exception as exc:
      print(f'函数执行报错：{exc}')
      raise

  return inner


@add_time_counter
def iter_fn(name: str, age: int):
    count = 1e7
    print(f'打印参数：{name} and {age}')
    print(f"count={count}")
    # err = count / 0 # 人为触发报错
    while count > 0:
        count -= 1

iter_fn(age=18, name='hhh')




打印参数：hhh and 18
count=10000000.0
执行函数耗时：0.29s


### 学习内置装饰器

- `@property`
  - 把方法转成属性
- `@classmethod`
  - 把方法绑定到类，接受 cls 参数，通常用于操作类本身
- `@staticmethod`
  - 定义不自动接受 `self` 或 `cls` 的方法，适合与类有逻辑关联的独立功能


In [45]:

class Person:
    __name = None

    def __init__(self, name=None) -> None:
        self.__name = name

    @property
    def get_name(self):
        print(f"self == Person: {self == Person}")
        print(
            "函数变成属性，只是调用时的形式发生了变化，本质还是一个函数，而且是被调用的时候才会被执行。相当于是  js 中的 getter"
        )
        return self.__name

    @classmethod
    def guest(cls):
        print(f"cls == Person: {cls == Person}")
        """
        cls 就是 Person 类
        cls 是 class 的简称
        """
        return cls("Guest")
    
    @staticmethod
    def is_valid_name(name):
      '''
      使用 staticmethod 装饰器后， is_valid_name 函数形参的首位就不用写 self 了。
      什么时候使用：当这个函数内部不需要使用 self 的时候
      '''
      return isinstance(name, str) and len(name) > 0


p1 = Person('xiaohong')
print(p1.get_name)

g1 = p1.guest()
print(g1.get_name)

g1.is_valid_name('test')


self == Person: False
函数变成属性，只是调用时的形式发生了变化，本质还是一个函数，而且是被调用的时候才会被执行。相当于是  js 中的 getter
xiaohong
cls == Person: True
self == Person: False
函数变成属性，只是调用时的形式发生了变化，本质还是一个函数，而且是被调用的时候才会被执行。相当于是  js 中的 getter
Guest


True

In [ ]:
import inspect


def test_params(*args, **kwargs):
    print(f"args={args}")
    print(f"kwargs={kwargs}")

test_params(1,2,3)
test_params(hello=1)

def test_params1(name, /, *, age):
  print(name, age)

test_params1('test', age=19)



sig = inspect.signature(test_params1)

print(sig.parameters)

for item in sig.parameters:
    print(type(sig.parameters[item]))
    # print(sig.parameters[item])



args=(1, 2, 3)
kwargs={}
args=()
kwargs={'hello': 1}
test 19
<class 'inspect.Signature'>
<class 'inspect.Parameter'>
<class 'inspect.Parameter'>
